# RAG 실험 결과 분석

이 노트북은 이미 실행된 RAG 실험 결과를 불러와 비교하고 시각화합니다.
직접 실험을 돌리지 않고 experiments/rag_*/ 폴더에 저장된 산출물만 읽습니다.

사용법: 아래 셀들을 순서대로 실행하세요. 실험 폴더를 자동으로 찾아서 분석합니다.

이 노트북은 실험 아티팩트로 남겨도 좋습니다. 실행 완료 시점의 결과 해석을 기록하는 용도로.

## 1. 실험 목록 불러오기

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    markers = ("AGENTS.md", "configs", "scripts", "src")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"

# metrics.json이 존재하는 실험 폴더만 자동 수집
EXPERIMENT_DIRS = sorted([
    d for d in EXPERIMENTS_ROOT.iterdir()
    if d.is_dir() and (d / "metrics.json").exists()
])

if not EXPERIMENT_DIRS:
    print("⚠ metrics.json이 있는 실험 폴더가 없습니다.")
    print("  먼저 run_rag_chat.py --evaluate 를 실행하세요.")
else:
    print(f"발견된 실험: {len(EXPERIMENT_DIRS)}개")
    for d in EXPERIMENT_DIRS:
        print(f"  {d.name}/")

print(f"\n프로젝트 루트: {PROJECT_ROOT}")

## 2. 실험 선택

비교할 실험을 선택합니다. 전체를 보려면 그대로 두고, 특정 실험만 보려면 리스트에서 제외하세요.

In [ ]:
# 전체 실험 보기: 그대로 두면 됨
# 특정 실험만: 포함할 폴더 이름을 직접 지정
# SELECTED = ["rag_langchain", "rag_hybrid"]  # <-- 수동 선택 예시

SELECTED = [d.name for d in EXPERIMENT_DIRS]
print(f"선택된 실험: {SELECTED}")

all_metrics = []
for exp_name in SELECTED:
    metrics_path = EXPERIMENTS_ROOT / exp_name / "metrics.json"
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text(encoding="utf-8"))
        m["experiment"] = exp_name
        all_metrics.append(m)

df_metrics = pd.DataFrame(all_metrics).set_index("experiment")
df_metrics

## 3. 지표 비교 - Bar Chart

모든 실험의 4개 지표를 나란히 비교합니다.

In [ ]:
if df_metrics.empty:
    print("표시할 데이터가 없습니다.")
else:
    metric_names = ["retrieval_hit_rate", "answer_contains_expected_rate",
                    "citation_correct_rate", "not_found_rate"]
    n_exp = len(df_metrics)
    fig, ax = plt.subplots(figsize=(max(8, n_exp * 1.8), 5))
    x = range(n_exp)
    bar_width = 0.2
    colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]
    for i, (metric, color) in enumerate(zip(metric_names, colors)):
        values = [df_metrics.loc[name, metric] if name in df_metrics.index else 0
                  for name in SELECTED]
        bars = ax.bar([pos + i * bar_width for pos in x], values, bar_width,
                      label=metric, color=color)
        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.02,
                        f"{val:.2f}", ha="center", fontsize=8)
    ax.set_ylabel("score")
    ax.set_title("RAG Metrics Comparison")
    ax.set_xticks([pos + bar_width * 1.5 for pos in x])
    ax.set_xticklabels(SELECTED, rotation=20, ha="right")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_ylim(0, 1.15)
    plt.tight_layout()
    plt.show()

## 4. 2개 실험 Before/After 집중 비교

두 실험을 골라서 지표 차이를 delta 값으로 보여줍니다.

In [ ]:
# 비교할 두 실험 이름을 지정 (자동으로 앞 2개)
if len(SELECTED) >= 2:
    A, B = SELECTED[0], SELECTED[1]
else:
    A, B = SELECTED[0], SELECTED[0]

print(f"Before: {A}")
print(f"After : {B}")
print()

if A in df_metrics.index and B in df_metrics.index:
    rows = []
    for metric in ["retrieval_hit_rate", "answer_contains_expected_rate",
                   "citation_correct_rate", "not_found_rate"]:
        a_val = df_metrics.loc[A, metric]
        b_val = df_metrics.loc[B, metric]
        delta = b_val - a_val
        emoji = "+ " if delta > 0.03 else "- " if delta < -0.03 else "= "
        rows.append({
            "metric": metric,
            A: f"{a_val:.3f}",
            B: f"{b_val:.3f}",
            "delta": f"{delta:+.3f} {emoji}",
        })
    display(pd.DataFrame(rows))
else:
    print("비교할 데이터가 부족합니다.")

## 5. Retriever 방식별 비교

reports/rag_retriever_comparison.csv 에서 retriever 방식별 성능을 불러옵니다.
이 파일은 compare_rag_retrievers.py 를 실행하면 자동 생성됩니다.

In [ ]:
COMPARISON_CSV = PROJECT_ROOT / "reports" / "rag_retriever_comparison.csv"

if COMPARISON_CSV.exists():
    df_comp = pd.read_csv(COMPARISON_CSV)
    display(df_comp)
    if not df_comp.empty:
        metric_cols = [c for c in ["retrieval_hit_rate",
                       "answer_contains_expected_rate", "citation_correct_rate"]
                       if c in df_comp.columns]
        if metric_cols and "experiment" in df_comp.columns:
            df_comp.set_index("experiment")[metric_cols].plot.bar(
                figsize=(10, 5),
                color=["#4CAF50", "#2196F3", "#FF9800"],
                title="Retriever Comparison"
            )
            plt.ylabel("score")
            plt.ylim(0, 1.1)
            plt.xticks(rotation=0)
            plt.tight_layout()
            plt.show()
else:
    print("rag_retriever_comparison.csv 없음")
    print("  python scripts/compare_rag_retrievers.py --project-root .")

## 6. 실패 분석 통합

모든 실험의 실패 유형을 모아서 봅니다.

In [ ]:
failure_summary = []
failure_types = ["bad_retrievals", "unsupported_answers", "failed_questions"]

for exp_name in SELECTED:
    row = {"experiment": exp_name}
    for ftype in failure_types:
        path = EXPERIMENTS_ROOT / exp_name / f"{ftype}.csv"
        if path.exists():
            try:
                df = pd.read_csv(path)
                row[ftype] = len(df)
            except Exception:
                row[ftype] = 0
        else:
            row[ftype] = 0
    failure_summary.append(row)

df_fail = pd.DataFrame(failure_summary).set_index("experiment")

if df_fail.empty or df_fail.sum().sum() == 0:
    print("모든 실험에서 실패 케이스가 없습니다!")
else:
    display(df_fail)
    df_fail.plot.bar(
        figsize=(10, 4),
        color=["#FF9800", "#2196F3", "#F44336"],
        title="Failure Case Breakdown by Experiment"
    )
    plt.ylabel("count")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## 7. 특정 실험 상세 진단

한 실험을 골라서 검색 결과와 실패 질문을 직접 확인합니다.

In [ ]:
DETAIL_EXP = SELECTED[0]
DETAIL_DIR = EXPERIMENTS_ROOT / DETAIL_EXP

print(f"분석 대상: {DETAIL_EXP}")
print()

# retrieval_results.jsonl 미리보기
ret_path = DETAIL_DIR / "retrieval_results.jsonl"
if ret_path.exists():
    print("=== 검색 결과 예시 (retrieval_results.jsonl) ===")
    lines = ret_path.read_text(encoding="utf-8").splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    for r in rows[:3]:
        q = r.get('question', '?')
        print(f"  질문: {q}")
        for chunk in r.get("retrieved_chunks", [])[:2]:
            cid = chunk.get('chunk_id', '?')
            scr = chunk.get('score', 0)
            txt = chunk.get('text', '')[:60]
            rk = chunk.get('rank', '?')
            print(f"    [{rk}등] score={scr:.3f}  {cid}: {txt}...")
        print()

# bad_retrievals.csv 확인
bad_path = DETAIL_DIR / "bad_retrievals.csv"
if bad_path.exists():
    bad = pd.read_csv(bad_path)
    if not bad.empty:
        print(f"=== 검색 실패 질문 ({len(bad)}건) ===")
        for _, row in bad.head(5).iterrows():
            print(f"  X {row.get('question', '?')}")
            print(f"    expected_answer: {str(row.get('expected_answer', '?'))[:50]}")

# answers.jsonl 상태 요약
ans_path = DETAIL_DIR / "answers.jsonl"
if ans_path.exists():
    lines = ans_path.read_text(encoding="utf-8").splitlines()
    answers = [json.loads(line) for line in lines if line.strip()]
    statuses = [a.get("status", "?") for a in answers]
    from collections import Counter
    print(f"\n=== 답변 상태 분포 ===")
    for status, count in Counter(statuses).items():
        print(f"  {status}: {count}건")

## 8. 노트북 저장 (아티팩트로 남기기)

분석이 끝난 후 File > Download as > HTML 로 내보내면 실험 리포트가 됩니다.

In [ ]:
print("실험 요약:")
print(f"  비교한 실험: {len(SELECTED)}개")
if not df_metrics.empty:
    best_idx = df_metrics["retrieval_hit_rate"].idxmax()
    best_val = df_metrics.loc[best_idx, "retrieval_hit_rate"]
    print(f"  최고 retrieval_hit_rate: {best_idx} = {best_val:.3f}")
print(f"  분석 일시: {pd.Timestamp.now()}")